In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import os
import pandas as pd

csv_path = os.path.join(path, "Q1_data.csv")
df = pd.read_csv(csv_path)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
#import pandas as pd
import matplotlib.pyplot as plt
def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(df, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:
df = df.drop(['Order_ID'],axis=1)

In [ ]:
df.head()

In [ ]:
# Task 2: Write your code here:
def check_missing_values(df):
  print("Original Data:")
  print(df)
  print("\nMissing values:")
  print(df.isnull().sum())


  df_drop = df.copy()
  df_drop['Delivery_Time']=df_drop['Delivery_Time'].dropna()

  df_mode = df.copy()
  df_mode['Weather'] = df_mode['Weather'].fillna(df_mode['Weather'].mode()[0])
  df_mode['Traffic_Level'] = df_mode['Traffic_Level'].fillna(df_mode['Traffic_Level'].mode()[0])
  df_mode['Time_of_Day'] = df_mode['Time_of_Day'].fillna(df_mode['Time_of_Day'].mode()[0])
  df_mode['Vehicle_Type'] = df_mode['Vehicle_Type'].fillna(df_mode['Vehicle_Type'].mode()[0])
  print("\n4. Fill with mode:")
  print(df_mode)
  df_mean = df.copy()
  df_mean['Distance_km'] = df_mean['Distance_km'].fillna(df_mean['Distance_km'].mean())
  df_mean['Preparation_Time_min'] = df_mean['Preparation_Time_min'].fillna(df_mean['Preparation_Time_min'].median())
  df['Courier_Experience_yrs'] = df['Courier_Experience_yrs'].dropna()
  print(df_mean)


check_missing_values(df)

In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import LabelEncoder
categorical_cols = df.select_dtypes(include=["object"]).columns

label_encoders = {}
for col in categorical_cols:
  le = LabelEncoder()
  df[col] = le.fit_transform(df[col])
  label_encoders[col] = le

df


In [ ]:
df['Delivery_Time'].isnull().sum()

In [ ]:
df.describe()

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler

numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
df.head()

In [ ]:
# Task 6: Write your code here:
def check_target_imbalance(df, target_column):
  print("Target Distribution:")
  print(df[target_column].value_counts(normalize=True))

check_target_imbalance(df, "Delivery_Time")
# since its imbal;anced we will use StratfiedKFold

In [ ]:
# Task 1: Write your code here:
X = df.drop("Delivery_Time", axis=1).astype(float)
y = df['Delivery_Time'].astype(float)
d = y.isnull().sum()
d

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.metrics import mean_absolute_error as sklearn_mae, r2_score
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestRegressor
import numpy as np
from tqdm import tqdm


def mean_absolute_error(y, y_hat):
  return (1 / (2 * len(y))) * np.sum(abs(y_hat - y))

def gradient_descent(X, y, learning_rate, n_iters=500):
  m, n = X.shape  # m rows, n columns (dimensions)
  theta = np.zeros(n)  # initialize a zeros weight vector with n dimensions
  losses = []

  for _ in tqdm(range(n_iters), desc="Training Linear Regression"):
    y_hat = np.dot(X, theta)
    gradient = np.dot(X.T, (y_hat - y)) / m
    theta -= learning_rate * gradient

    loss = mean_absolute_error(y, y_hat)
    losses.append(loss)

  return theta, losses

n_splits = 5
skf= StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

lr_losses = []
lr_mae = []
lr_r2 = []

for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # Train
  theta, losses = gradient_descent(X_train.values, y_train.values, learning_rate=0.1, n_iters=500)

  # Validate
  y_pred = np.dot(X_test.values, theta)

  # Calculate evaluation metrics
  mae = sklearn_mae(y_test, y_pred)
  r2 = r2_score(y_test, y_pred)

  # Store results
  lr_losses.append(losses)
  lr_mae.append(mae)
  lr_r2.append(r2)

average_losses = np.mean(lr_losses, axis=0)
print("Linear Regression Results")
print(f"  Average MSE: {np.mean(lr_mae):.4f}")
print(f"  Average R2:  {np.mean(lr_r2):.4f}")

In [ ]:
# Task 1: Write your code here:
from sklearn.linear_model import Ridge, Lasso

models = {
  "Ridge Regression": Ridge(alpha=1.0, max_iter=10000),
  "LASSO Regression": Lasso(alpha=1.0,  max_iter=10000),}

all_results = {}

for name in models:
  all_results[name] = {'mae': [], 'r2': []}


In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: